In [1]:
import json
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("data/output")

conversations = []
analysis_rows = []

def load_json(path):
    for enc in ("utf-8", "cp1252"):
        try:
            with open(path, encoding=enc) as f:
                return json.load(f)
        except (UnicodeDecodeError, json.JSONDecodeError):
            continue
    raise ValueError(f"Could not decode {path}")

for path in sorted(OUTPUT_DIR.glob("conversation_*.json")):
    data = load_json(path)

    patient_id = data["id"]
    pattern = data.get("conversation_pattern", {})
    nodes = {f"node_{k}": v for k, v in data.get("nodes", {}).items()}
    edges = {f"edge_{k}": v for k, v in data.get("edges", {}).items()}

    for turn in data["conversation"]:
        conversations.append({
            "patient_id": patient_id,
            "turn_index": turn["turn_index"],
            "speaker": turn["speaker"],
            "text": turn["text"],
            **pattern,
            **nodes,
            **edges,
        })

    for item in data.get("Analysis", []):
        analysis_rows.append({
            "patient_id": patient_id,
            "feature": item["feature_name"],
            "predicted_score": item["score"],
            "explanation": item["explanation"],
            "ground_truth": (
                data["nodes"].get(item["feature_name"])
                or data["edges"].get(item["feature_name"])
            ),
        })

df_conv = pd.DataFrame(conversations)
df_analysis = pd.DataFrame(analysis_rows)

print(f"Conversations: {len(df_conv)} rows")
print(f"Analysis:      {len(df_analysis)} rows")

Conversations: 154 rows
Analysis:      0 rows


In [2]:
df_conv

,patient_id,turn_index,speaker,text,emotion,intensity,expressiveness,disclosure,politeness,assertiveness,...,edge_Threat -- Memory,edge_Threat -- Negative appraisals,edge_Threat -- Triggers,edge_Maladaptive Strategies -- Memory,edge_Maladaptive Strategies -- Negative appraisals,edge_Triggers -- Negative Appraisals,edge_Negative Appraisals -- Maladaptive Strategies,edge_Negative Appraisals -- Threat,edge_Negative Appraisals -- Memory,edge_Negative Appraisals -- Triggers
0,1,0,bot,"Hello, how are you?",anxious,severe,low,low,formal,high,...,0,0,0,0,0,0,0,1,0,0
1,1,1,client,"I appreciate your inquiry. Recently, I was in ...",anxious,severe,low,low,formal,high,...,0,0,0,0,0,0,0,1,0,0
2,1,2,bot,Thank you for sharing that experience. In that...,anxious,severe,low,low,formal,high,...,0,0,0,0,0,0,0,1,0,0
3,1,3,client,"In that moment, my heart started racing, and I...",anxious,severe,low,low,formal,high,...,0,0,0,0,0,0,0,1,0,0
4,1,4,bot,How did your thoughts about the situation chan...,anxious,severe,low,low,formal,high,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,7,5,bot,That sounds really taxing. Have you noticed an...,angry,severe,low,low,casual,high,...,0,0,0,0,0,0,0,0,0,0
150,7,6,client,"Yeah, I can’t shake the thought that something...",angry,severe,low,low,casual,high,...,0,0,0,0,0,0,0,0,0,0
151,7,7,bot,That sounds overwhelming to deal with. Have yo...,angry,severe,low,low,casual,high,...,0,0,0,0,0,0,0,0,0,0
152,7,8,client,Definitely. I’ve been avoiding crowded places;...,angry,severe,low,low,casual,high,...,0,0,0,0,0,0,0,0,0,0


In [3]:
df_conv.to_csv(OUTPUT_DIR / "conversations.csv", index=False, encoding="utf-8")
print("Saved conversations.csv")

Saved conversations.csv
